In [ ]:
# NLGCL Kaggle Notebook
# This notebook runs the NLGCL model on the Kaggle platform.
# It automatically sets up the environment, downloads the dataset, and runs training.

import os
import sys
import shutil

# 1. Setup Environment & Clone
if not os.path.exists('NLGCL') and not os.path.exists('recbole_gnn'):
    !git clone https://github.com/Jinfeng-Xu/NLGCL.git
    os.chdir('NLGCL')
else:
    if os.path.basename(os.getcwd()) != 'NLGCL' and os.path.exists('NLGCL'):
        os.chdir('NLGCL')

sys.path.append(os.getcwd())
print(f"Working Directory: {os.getcwd()}")

# 2. Install Dependencies
# We use a loop to ensure recbole is actually installed and importable
import subprocess

def install_package(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

try:
    import recbole
    print(f"RecBole version {recbole.__version__} is already installed.")
except ImportError:
    print("RecBole not found. Installing...")
    !pip install recbole
    # Also install other requirements
    if os.path.exists('requirements.txt'):
         !pip install -r requirements.txt
    
    # GNN libraries
    !pip install torch-scatter torch-sparse torch-cluster torch-spline-conv torch-geometric -f https://data.pyg.org/whl/torch-2.0.0+cu118.html
    
    # Force re-check
    try:
        import recbole
        print("RecBole installed successfully.")
    except ImportError:
        print("CRITICAL: RecBole installation failed or path not updated. Please RESTART KERNEL.")

# 3. Dataset Preparation
dataset_name = 'Yelp'
target_dir = f'dataset/{dataset_name}'

# Check /kaggle/input
if not os.path.exists(target_dir) and os.path.exists('/kaggle/input'):
    print("Searching /kaggle/input...")
    for root, dirs, files in os.walk('/kaggle/input'):
        if f'{dataset_name.lower()}.inter' in files or dataset_name in root.split(os.sep):
            print(f"Found in {root}. Copying...")
            shutil.copytree(root, target_dir, dirs_exist_ok=True)
            break

if not os.path.exists(target_dir):
    print(f"Downloading {dataset_name}...")
    import requests, zipfile
    url = "https://recbole.s3-accelerate.amazonaws.com/ProcessedDatasets/Yelp/yelp.zip"
    
    if not os.path.exists(target_dir): os.makedirs(target_dir)
    zip_path = os.path.join(target_dir, f"{dataset_name}.zip")
    
    try:
        with requests.get(url, stream=True) as r:
            r.raise_for_status()
            with open(zip_path, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192): f.write(chunk)
        with zipfile.ZipFile(zip_path, 'r') as z: z.extractall(target_dir)
        os.remove(zip_path)
        print("Download complete.")
    except Exception as e:
        print(f"Download failed: {e}")

# 4. Run Training
# We import inside the function or block to ensure path updates are caught
import torch
from logging import getLogger
from recbole.utils import init_logger, init_seed, set_color
from recbole_gnn.config import Config
from recbole_gnn.utils import create_dataset, data_preparation, get_model, get_trainer

def run_single_model(args):
    config = Config(model=args.model, dataset=args.dataset, config_file_list=args.config_file_list)
    if 'data_path' not in config: config['data_path'] = 'dataset/'

    init_seed(config['seed'], config['reproducibility'])
    init_logger(config)
    logger = getLogger()
    logger.info(config)

    dataset = create_dataset(config)
    logger.info(dataset)

    train_data, valid_data, test_data = data_preparation(config, dataset)

    model = get_model(config['model'])(config, train_data.dataset).to(config['device'])
    logger.info(model)

    trainer = get_trainer(config['MODEL_TYPE'], config['model'])(config, model)

    best_valid_score, best_valid_result = trainer.fit(train_data, valid_data, saved=True, show_progress=config['show_progress'])
    test_result = trainer.evaluate(test_data, load_best_model=True, show_progress=config['show_progress'])

    logger.info(set_color('best valid ', 'yellow') + f': {best_valid_result}')
    logger.info(set_color('test result', 'yellow') + f': {test_result}')

class Args:
    def __init__(self):
        self.model = 'NLGCL'
        self.dataset = 'Yelp'
        self.config_files = ''
        self.config_file_list = ['properties/overall.yaml']

if os.path.exists(f'dataset/{Args().dataset}'):
    run_single_model(Args())
else:
    print("Dataset missing.")